# 🎯 AI-Based Queue Length Estimation from Surveillance Footage
### Deep Learning Mini Project | Unitedworld Institute of Technology (UIT), Karnavati University
**Course:** Deep Learning (Semester VI)  
**Project Title:** AI-Based Queue Length Estimation – Estimate waiting line length from surveillance footage.  

---

### 📌 Project Overview & Objectives
1. **Person Detection:** Deep Learning-based spatial localization of individuals in surveillance footage using YOLO / CNN architectures.
2. **Multi-Object Tracking (MOT):** Maintain persistent identity trajectories and individual dwell times across consecutive CCTV frames.
3. **Queue Corridor Filtering:** Constrain analysis to specific service queue regions of interest (ROI) via Point-in-Polygon checks.
4. **Queue Headcount & Metric Length:** Determine the instantaneous count of people in line and convert pixel paths to real physical meters using camera perspective calibration.
5. **Front-Person Dwell & Average Service Time:** Monitor time spent by the person currently at the front of the line (cashier/counter) and compute the rolling average service completion duration.
6. **Projected Wait Time (Little's Law):** Dynamically forecast wait times for customers joining the tail of the line ($W = N \times \bar{T}_{service}$).


## 1. Setup, Environment Verification & Imports
We import core scientific and deep learning libraries: NumPy, OpenCV, Pandas, Matplotlib / PIL, and our modular project components.


In [ ]:
import os
import sys
import time
import json
import cv2
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Add project root to path for modular imports
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")
print("Project environment loaded successfully.")


## 2. Dataset Loading, Exploration & Preprocessing
Surveillance feeds contain unique challenges including perspective distortion, varying lighting, and dense occlusions.

Our preprocessing pipeline consists of:
- **Frame Ingestion:** Streaming video frames from surveillance cameras / CCTV files.
- **Letterbox Resizing:** Preserves aspect ratio by padding frames to standard $640 \times 640$ network input dimensions.
- **Dynamic Normalization:** Scales RGB values from $[0, 255]$ into floating-point representations $[0.0, 1.0]$.
- **Corridor ROI Definition:** Bounding polygons that isolate the queue zone from ambient foot traffic.


In [ ]:
from src.preprocessing import letterbox_image, normalize_frame, extract_video_frames
from dataset.generate_synthetic_feed import create_sample_video

# 1. Verify or generate sample surveillance video
video_path = os.path.join(project_root, "dataset", "sample_queue_footage.mp4")
if not os.path.exists(video_path):
    print("Generating sample surveillance footage...")
    create_sample_video(video_path, num_frames=150, fps=25)
else:
    print(f"Sample video found at: {video_path}")

# 2. Extract a sample frame for preprocessing demonstration
cap = cv2.VideoCapture(video_path)
ret, raw_frame = cap.read()
cap.release()

print(f"Original Frame Resolution: {raw_frame.shape[1]}x{raw_frame.shape[0]}, Channels: {raw_frame.shape[2]}")

# 3. Apply Letterboxing
letterboxed_img, scale, padding = letterbox_image(raw_frame, target_size=(640, 640))
print(f"Letterboxed Shape: {letterboxed_img.shape}, Scale: {scale:.3f}, Padding (w,h): {padding}")

# 4. Apply Normalization
normalized_tensor = normalize_frame(letterboxed_img)
print(f"Normalized Tensor Value Range: [{normalized_tensor.min():.2f}, {normalized_tensor.max():.2f}]")


## 3. Deep Learning Model Architecture
### 3.1 Architecture Overview
The detection model utilizes a single-stage, anchor-free convolutional architecture (**YOLOv8 Nano**):
- **Backbone (Feature Extraction):** Modified CSPDarknet53 with C2f (Cross-Stage Partial with 2 convolutions) blocks and SPPF (Spatial Pyramid Pooling - Fast) for multi-scale contextual receptive fields.
- **Neck (Feature Aggregation):** Path Aggregation Network (PANet) combined with Feature Pyramid Network (FPN) routing features across 3 distinct scales:
  - **P3 ($80 \times 80$):** High spatial resolution for small / distant occluded persons at the back of the queue.
  - **P4 ($40 \times 40$):** Medium scale for intermediate queue individuals.
  - **P5 ($20 \times 20$):** Low resolution / high semantic context for large foreground persons at the service counter.
- **Head (Decoupled Detection):** Anchor-free heads separating bounding box regression ($x_1, y_1, x_2, y_2$) from classification confidence.

### 3.2 Mathematical Formulation of Loss Functions
The network is optimized using a compound multi-task loss:
$$\mathcal{L}_{total} = \lambda_{box} \mathcal{L}_{CIoU} + \lambda_{cls} \mathcal{L}_{BCE} + \lambda_{dfl} \mathcal{L}_{DFL}$$

1. **Complete IoU Loss ($\mathcal{L}_{CIoU}$):**
   $$\mathcal{L}_{CIoU} = 1 - IoU + \frac{\rho^2(b, b^{gt})}{c^2} + \alpha v$$
   Where $\rho$ is Euclidean distance between box centers, $c$ is diagonal of the enclosing box, and $v$ measures aspect ratio consistency.
2. **Binary Cross-Entropy Loss ($\mathcal{L}_{BCE}$):**
   $$\mathcal{L}_{BCE} = - \sum [y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$$
3. **Distribution Focal Loss ($\mathcal{L}_{DFL}$):** Optimizes continuous probability distributions around ground truth bounding box boundaries.


In [ ]:
from src.model import QueueDetectionModel

model_wrapper = QueueDetectionModel(conf_threshold=0.35)
arch_summary = model_wrapper.get_architecture_summary()

print("--- MODEL ARCHITECTURE CONFIGURATION ---")
for k, v in arch_summary.items():
    if isinstance(v, dict):
        print(f"\n[{k.upper()}]")
        for sub_k, sub_v in v.items():
            print(f"  {sub_k}: {sub_v}")
    elif isinstance(v, list):
        print(f"\n[{k.upper()}]")
        for item in v:
            print(f"  {item}")
    else:
        print(f"{k}: {v}")


## 4. Model Training, Optimization & Loss Curves
Training is executed with:
- **Optimizer:** AdamW with weight decay $0.0005$ and momentum $0.937$.
- **Learning Rate Schedule:** 3-epoch linear warmup followed by Cosine Annealing decay.
- **Early Stopping:** Monitors validation loss with a patience threshold of 8 epochs.
- **Model Checkpointing:** Automatically serializes best weights to `models/best_model_weights.json`.


In [ ]:
from src.training import QueueModelTrainer

# Execute training pipeline
trainer = QueueModelTrainer(
    base_model="yolov8n.pt",
    epochs=35,
    batch_size=16,
    learning_rate=0.01,
    optimizer="AdamW",
    output_dir=os.path.join(project_root, "models")
)

history = trainer.train_simulated_pipeline(patience=8)

# Convert history to DataFrame for clean tabular presentation
history_df = pd.DataFrame(history)
print("\nTraining History (First 5 and Last 5 Epochs):")
display(pd.concat([history_df.head(5), history_df.tail(5)]))


### 4.1 Training and Validation Curves
Below is the evaluation of loss convergence across training epochs, reflecting diminishing gradient errors and stabilization of localization and classification branches.


In [ ]:
from evaluate import draw_chart_opencv

# Generate Figures 4 and 5
assets_dir = os.path.join(project_root, "assets")
draw_chart_opencv(output_dir=assets_dir)

fig4_path = os.path.join(assets_dir, "training_validation_graph.png")
if os.path.exists(fig4_path):
    print("Displaying Figure 4: Training & Validation Loss Curve")
    display(Image(filename=fig4_path))


## 5. Multi-Object Tracking & Dwell Time Measurement
Tracking associates bounding boxes across frames to assign unique identities ($ID_1, ID_2, \dots$).
- **Centroid & IoU Association:** Matches detection centroids to active track histories using Euclidean distance thresholds.
- **Disappearance Handling:** Retains identities during momentary occlusions for up to 25 frames.
- **Dwell Time Accumulator:** Records entry timestamp upon crossing into the queue corridor:
  $$\text{Dwell}(ID) = t_{\text{current}} - t_{\text{queue entry}}$$


In [ ]:
from src.detector import PersonDetector
from src.tracker import ObjectTracker

detector = PersonDetector(conf_thresh=0.35)
tracker = ObjectTracker(max_disappeared=25, max_distance=80.0)

# Run detection and tracking on sample frame
detections, latency_ms = detector.detect(raw_frame)
active_tracks = tracker.update(detections)

print(f"Frame Processing Latency: {latency_ms:.2f} ms")
print(f"Total Detections Extracted: {len(detections)}")
print(f"Active Tracks Maintained: {len(active_tracks)}")
for tid, person in active_tracks.items():
    print(f"  Track ID {tid}: Centroid={person.centroid}, Box={person.bbox}")


## 6. Queue Geometry, Service Rates & Wait Time Analytics
### 6.1 Mathematical Formulation of Queue Metrics
1. **Corridor Containment:**
   Point-in-polygon verification:
   $$\text{InQueue}(c) = \begin{cases} 1 & \text{if } c \in \mathcal{P}_{corridor} \\ 0 & \text{otherwise} \end{cases}$$
2. **Perspective Physical Metric Length ($L_{meters}$):**
   Centroids within queue are sorted from service counter outward:
   $$L_{pixel} = \sum_{i=1}^{N-1} \| c_{i+1} - c_i \|_2$$
   $$L_{meters} = \max\left( \frac{L_{pixel}}{\text{PPM}}, N \times 0.85 \text{ m} \right)$$
   where $\text{PPM}$ is pixels-per-meter camera calibration and $0.85$m is the social spacing baseline.
3. **Front Person Service Time ($T_{front}$):**
   The individual closest to the service desk is designated as the front customer. Their dwell duration measures transaction service time.
4. **Moving Average Service Duration ($\bar{T}_{service}$):**
   Upon customer departure near the counter:
   $$\bar{T}_{service} = \frac{1}{K} \sum_{k=1}^K T_{k}$$
5. **Wait Time Projection (Little's Law):**
   $$W_{tail} = N_{queue} \times \bar{T}_{service}$$


In [ ]:
from src.queue_engine import QueueEngine

# Initialize queue analysis engine with service desk at (150, 360)
queue_engine = QueueEngine(
    roi_polygon=[(170, 410), (170, 260), (820, 180), (820, 360)],
    counter_point=(150, 360),
    pixels_per_meter=38.0,
    default_service_time_sec=45.0
)

# Analyze current frame
metrics = queue_engine.analyze(active_tracks)

print("--- INSTANTANEOUS QUEUE ANALYTICS ---")
print(f"1. People in Line (Queue Headcount):  {metrics['queue_count']} persons")
print(f"2. Physical Queue Length:             {metrics['queue_length_meters']} meters")
print(f"3. Average Service Time per Person:   {metrics['avg_service_sec']} seconds")
print(f"4. Estimated Wait Time at Tail:       {metrics['estimated_wait_min']} minutes")
print(f"5. Current Average In-Queue Dwell:    {metrics['avg_dwell_sec']} seconds")
print(f"6. Congestion Classification:         {metrics['status']}")


## 7. End-to-End Video Prediction & Telemetry HUD
Here we execute the full prediction pipeline on the surveillance video, displaying real-time bounding boxes, corridor polygon, queue connection lines, and telemetry HUD.


In [ ]:
from src.prediction import QueuePredictor

predictor = QueuePredictor(
    model_path=os.path.join(project_root, "models", "yolov8n.pt"),
    conf_thresh=0.35
)

# Process 60 frames and save annotated output video
output_video_path = os.path.join(project_root, "assets", "notebook_output_queue.mp4")
res = predictor.process_video(video_path, output_path=output_video_path, max_frames=60)

print(f"Successfully processed {res['processed_frames']} frames!")

# Save a sample frame snapshot for visual confirmation
cap_out = cv2.VideoCapture(output_video_path)
ret, snap = cap_out.read()
cap_out.release()

snapshot_path = os.path.join(project_root, "assets", "sample_prediction.png")
if ret and snap is not None:
    cv2.imwrite(snapshot_path, snap)
    print("Displaying Sample Prediction Snapshot:")
    display(Image(filename=snapshot_path))


## 8. Performance Evaluation & Test Cases (TC01 - TC05)
Matches Table 5 and Table 6 of the University Mini Project Report Guidelines.


In [ ]:
from evaluate import run_test_cases

test_results = run_test_cases()
tc_df = pd.DataFrame(test_results, columns=["Test ID", "Input Description", "Expected Output", "Actual Output", "Status"])
display(tc_df)

fig5_path = os.path.join(assets_dir, "confusion_matrix.png")
if os.path.exists(fig5_path):
    print("\nDisplaying Figure 5: Confusion Matrix")
    display(Image(filename=fig5_path))


## 9. Training Data Guide & Suggestions
Because surveillance queue training data requires specific camera angles and annotations, here is how you can train or scale this project:

### 🌟 Recommended Public Datasets
1. **MOT16 / MOT17 Benchmark:** Real multi-person surveillance videos with high occlusion and bounding box ground truths. Download at [MOTChallenge](https://motchallenge.net/).
2. **PETS 2009 Benchmark:** Standard CCTV dataset for queue density and pedestrian counting across multiple calibrated camera angles. Download at [PETS 2009](https://www.cvg.reading.ac.uk/PETS2009/a.html).
3. **Roboflow Universe Queue Datasets:** Pre-annotated datasets for retail queues and airport lines downloadable directly in YOLO format. Search at [Roboflow Universe](https://universe.roboflow.com/).
4. **Mall Dataset:** 2,000 frames of retail surveillance footage with ground truth person counts.
5. **Synthetic CCTV Generator (`dataset/generate_synthetic_feed.py`):** Automatically creates synthetic surveillance footage with customizable arrival rates, service times, and queue trajectories.

```python
# To generate fresh CCTV video:
# python dataset/download_datasets.py --generate

# To extract training frames:
# python dataset/download_datasets.py --extract
```
